# 15 · Training Data — Engineered, Not Found

**Part 2 · Machine Learning Systems** · *Prerequisites: 13–14* · *Book mapping: DMLS ch. 4*

No dataset is "the data". Someone chose what to log, how to sample it, what counts as a
label, and what to do about the class nobody sees. Each choice is invisible in the final
`X, y` arrays — and each one bounds what any model trained on them can be. This notebook
makes those choices visible and tests them.

**Learning objectives.**

1. Demonstrate sampling bias, and build the two workhorses: **stratified** and
   **reservoir** sampling (streams of unknown length, one pass).
2. Face labeling reality: annotator disagreement, **weak supervision** with labeling
   functions, and **active learning** under a labeling budget — all measured against
   hand-labeling.
3. Reproduce the **accuracy trap** of class imbalance and fix it three ways, choosing
   metrics that can't lie about rare classes.
4. Use **augmentation** to buy robustness with compute instead of collection.

In [ ]:
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_recall_curve,
                             precision_score, recall_score)
from sklearn.tree import DecisionTreeClassifier

from sysdes import viz

viz.use_style()
rng = np.random.default_rng(seed=15)

## 1. Sampling — where bias is born

### 1.1 Failure first: convenience sampling

The team needs "average weekly sessions per user" to size a feature. The quick query grabs
last week's *active* users — the ones conveniently in the activity table. But activity is
exactly what's being measured; sampling on it is selecting on the outcome:

In [ ]:
N = 100_000
weekly_sessions = rng.gamma(shape=0.9, scale=4.0, size=N)      # most users near zero
true_mean = weekly_sessions.mean()

# 'Convenient' sample: users who showed up this week - P(appear) grows with usage.
p_appear = 1 - np.exp(-weekly_sessions / 4)
convenient = weekly_sessions[rng.random(N) < p_appear]
uniform = rng.choice(weekly_sessions, size=len(convenient), replace=False)

print(f"true mean {true_mean:.2f} | uniform sample {uniform.mean():.2f} "
      f"| convenience sample {convenient.mean():.2f} "
      f"({convenient.mean()/true_mean - 1:+.0%})")
assert abs(uniform.mean() - true_mean) < 0.1
assert convenient.mean() > true_mean * 1.35, \
    "sampling on activity inflated the estimate ~50% - and no sample size fixes bias"

The punchline is the last clause: variance shrinks with more data; **bias doesn't**. Every
"we trained on the users we had" is a version of this cell.

### 1.2 Stratified sampling — guarantee the rare slice

Uniform sampling represents rare groups *on average* but not *every time* — a 3% class in
a 200-row sample will sometimes have 1 row, sometimes 12. **Stratified** sampling fixes
each group's share by sampling within groups:

In [ ]:
labels = (rng.random(20_000) < 0.03).astype(int)               # 3% positive class

def random_share(k=200):
    return labels[rng.choice(len(labels), k, replace=False)].mean()

def stratified_share(k=200):
    pos = np.where(labels == 1)[0]
    neg = np.where(labels == 0)[0]
    k_pos = round(k * 0.03)
    take = np.concatenate([rng.choice(pos, k_pos, replace=False),
                           rng.choice(neg, k - k_pos, replace=False)])
    return labels[take].mean()

rand_shares = [random_share() for _ in range(300)]
strat_shares = [stratified_share() for _ in range(300)]
print(f"positive share across 300 samples - random: {np.mean(rand_shares):.3f} "
      f"± {np.std(rand_shares):.3f}, stratified: {np.mean(strat_shares):.3f} "
      f"± {np.std(strat_shares):.3f}")
assert np.std(strat_shares) < np.std(rand_shares) / 5, \
    "stratification removes the sampling lottery for rare groups"

This is why train/test splits are stratified by default, and why evaluation *slices*
(nb 17) get their own quota: a metric on 3 rare-class examples is a coin flip.

### 1.3 Reservoir sampling — a fair sample from an endless stream

Streams (nb 11) have no `len()`. **Reservoir sampling** keeps a uniform k-sample in one
pass and O(k) memory: fill the reservoir with the first k items; thereafter item *i*
enters with probability k/i, evicting a random resident:

In [ ]:
def reservoir_sample(stream, k: int, seed: int) -> list:
    r = np.random.default_rng(seed)
    reservoir = []
    for i, item in enumerate(stream, start=1):
        if i <= k:
            reservoir.append(item)
        else:
            j = int(r.integers(0, i))       # uniform in [0, i)
            if j < k:                        # prob k/i: evict a random resident
                reservoir[j] = item
    return reservoir

# Verify UNIFORMITY empirically: every position's inclusion rate must be ~k/n.
n, k, trials = 10_000, 100, 400
hits = np.zeros(n)
for t in range(trials):
    for item in reservoir_sample(range(n), k, seed=t):
        hits[item] += 1
rates = hits / trials
early, late = rates[:100].mean(), rates[-100:].mean()
print(f"inclusion rate - target {k/n:.3f}; early items {early:.4f}, late items {late:.4f}")
assert abs(early - k / n) < 0.003 and abs(late - k / n) < 0.003, \
    "one pass, unknown length, still uniform - early and late items equally likely"

(The weighted variant powers "sample 1% of traffic, but 100% of errors" logging policies —
exercise 2 probes what that does to training.)

## 2. Labels — the expensive fiction

### 2.1 Annotators disagree (label multiplicity)

Send the same 1,000 support tickets to three annotators with instructions to mark
"urgent". Each annotator applies a slightly different internal rule — that's not
sloppiness, it's the ambiguity of the concept surfacing:

In [ ]:
URGENT_WORDS = ["outage", "down", "urgent", "asap", "broken", "failing",
                "crash", "emergency", "blocked", "critical", "severe", "offline"]
CALM_WORDS = ["question", "how", "thanks", "feature", "wondering", "docs",
              "pricing", "invoice", "tutorial", "feedback", "suggestion", "curious"]
NEUTRAL_WORDS = [f"word{i:03d}" for i in range(300)]   # mundane filler vocabulary

def make_ticket(urgent: bool) -> str:
    """A ticket = mostly filler + (usually) one or two class-indicative words.
    15% carry no signal at all; 10% of signals come from the WRONG list
    (quotes, sarcasm, 'is there an outage? just curious') - irreducible noise."""
    words = list(rng.choice(NEUTRAL_WORDS, size=8))
    if rng.random() < 0.85:
        vocab = URGENT_WORDS if urgent else CALM_WORDS
        if rng.random() < 0.10:
            vocab = CALM_WORDS if urgent else URGENT_WORDS
        words += list(rng.choice(vocab, size=int(rng.integers(1, 3))))
    bang = "!!" if (urgent and rng.random() < 0.5) else ""
    return " ".join(words) + bang

y_true = (rng.random(1_000) < 0.3).astype(int)
tickets = [make_ticket(bool(t)) for t in y_true]

def annotate(text, true, *, noise, strict):
    """An annotator: mostly follows the truth, flips with `noise`, and a
    'strict' annotator also marks any '!!' ticket urgent regardless."""
    label = true if rng.random() > noise else 1 - true
    if strict and "!!" in text:
        label = 1
    return label

ann = pd.DataFrame({
    "a1": [annotate(t, y, noise=0.08, strict=False) for t, y in zip(tickets, y_true)],
    "a2": [annotate(t, y, noise=0.12, strict=False) for t, y in zip(tickets, y_true)],
    "a3": [annotate(t, y, noise=0.05, strict=True) for t, y in zip(tickets, y_true)],
})
disagree = (ann.nunique(axis=1) > 1).mean()
majority = ann.mode(axis=1)[0].astype(int)
print(f"tickets with at least one disagreement: {disagree:.0%}")
print(f"accuracy - a1: {accuracy_score(y_true, ann.a1):.3f}, "
      f"a2: {accuracy_score(y_true, ann.a2):.3f}, a3: {accuracy_score(y_true, ann.a3):.3f}, "
      f"majority vote: {accuracy_score(y_true, majority):.3f}")
assert disagree > 0.15, "disagreement is the norm, not an accident"
assert accuracy_score(y_true, majority) >= max(
    accuracy_score(y_true, ann[c]) for c in ann) - 0.005, \
    "majority vote matches or beats every individual annotator"

Operational consequences: measure inter-annotator agreement *before* trusting labels;
make the labeling guide executable (a3's "!!" rule is a guideline bug you can now *see*);
and route systematic disagreements to adjudication — they usually mean the label
definition (nb 13 §2) is underspecified, not that annotators are bad.

### 2.2 Weak supervision — programmatic labels at scale

Hand labels: high quality, tiny volume, weeks of latency. **Weak supervision** flips the
trade: encode annotator heuristics as **labeling functions** (LFs) — noisy, partial
programs that vote (or abstain) — combine their votes, and label *everything you have*
instantly. The model then trains on many noisy labels instead of few clean ones:

In [ ]:
ABSTAIN = -1
def lf_urgent_words(t):  # decent coverage, hurt by the 10% wrong-list noise
    return 1 if any(w in t for w in URGENT_WORDS[:8]) else ABSTAIN
def lf_calm_words(t):
    return 0 if any(w in t for w in CALM_WORDS) else ABSTAIN
def lf_exclamation(t):   # a3's bias from §2.1, now an inspectable function
    return 1 if "!!" in t else ABSTAIN
def lf_severe(t):
    return 1 if re.search(r"severe|critical|emergency", t) else ABSTAIN

LFS = [lf_urgent_words, lf_calm_words, lf_exclamation, lf_severe]

# A big pile of UNLABELED tickets (labels exist only to grade ourselves later):
y_pool = (rng.random(5_000) < 0.3).astype(int)
pool = [make_ticket(bool(t)) for t in y_pool]

votes = np.array([[lf(t) for lf in LFS] for t in pool])
coverage = (votes != ABSTAIN).any(axis=1)
weak_label = np.array([np.bincount(row[row != ABSTAIN], minlength=2).argmax()
                       for row in votes[coverage]])
print(f"LF coverage: {coverage.mean():.0%}; "
      f"weak-label accuracy where covered: "
      f"{accuracy_score(y_pool[coverage], weak_label):.3f}")

# Grade the trade: 100 perfect hand labels vs thousands of ~88%-accurate weak ones.
y_test = (rng.random(2_000) < 0.3).astype(int)
test_txt = [make_ticket(bool(t)) for t in y_test]
vec = CountVectorizer()
vec.fit(pool + test_txt)

hand_model = LogisticRegression(max_iter=1000).fit(
    vec.transform(pool[:100]), y_pool[:100])                 # the hand-label budget
weak_model = LogisticRegression(max_iter=1000).fit(
    vec.transform([t for t, c in zip(pool, coverage) if c]), weak_label)

acc_hand = accuracy_score(y_test, hand_model.predict(vec.transform(test_txt)))
acc_weak = accuracy_score(y_test, weak_model.predict(vec.transform(test_txt)))
print(f"test accuracy - 100 hand labels: {acc_hand:.3f}, "
      f"{coverage.sum():,} weak labels: {acc_weak:.3f}")
assert acc_weak > acc_hand, \
    "volume of noisy labels beat scarcity of perfect ones - the weak supervision bet"

Why this matters beyond the accuracy delta: LFs are **code** — versioned, debuggable,
re-runnable when the label definition changes (change the LFs, relabel 5M tickets in
minutes — nb 12's refold, for labels). Real label models (Snorkel) go beyond majority
vote, learning each LF's accuracy and correlations from their agreement patterns
(exercise 4).

### 2.3 Active learning — spend the budget where the model is confused

When hand-labeling is the only option, *choose* what to label. **Uncertainty sampling**:
train on what you have, ask annotators about the points the model is least sure of
(p ≈ 0.5), repeat:

In [ ]:
def blobs(n, seed):
    r = np.random.default_rng(seed)
    y = (r.random(n) < 0.5).astype(int)
    X = r.normal(0, 1.15, (n, 2)) + np.where(y[:, None] == 1, [1.6, 1.2], [0, 0])
    return X, y

def run_budget(strategy: str, seed: int, initial=20, rounds=6, per_round=25):
    X_pool, y_pool = blobs(3_000, seed)
    X_test, y_test = blobs(2_000, seed + 999)
    labeled = list(range(initial))
    for _ in range(rounds):
        model = LogisticRegression().fit(X_pool[labeled], y_pool[labeled])
        candidates = np.setdiff1d(np.arange(len(X_pool)), labeled)
        if strategy == "random":
            pick = candidates[:per_round]
        else:  # uncertainty: closest to the decision boundary
            p = model.predict_proba(X_pool[candidates])[:, 1]
            pick = candidates[np.argsort(np.abs(p - 0.5))[:per_round]]
        labeled += list(pick)
    model = LogisticRegression().fit(X_pool[labeled], y_pool[labeled])
    return accuracy_score(y_test, model.predict(X_test))

seeds = range(8)
acc_random = np.mean([run_budget("random", s) for s in seeds])
acc_active = np.mean([run_budget("uncertainty", s) for s in seeds])
print(f"170-label budget, mean over 8 runs - random: {acc_random:.3f}, "
      f"uncertainty sampling: {acc_active:.3f}")
assert acc_active > acc_random, \
    "the same budget buys more accuracy when spent at the decision boundary"

The production loop version: yesterday's low-confidence (or disagreed-upon, or drifted —
nb 19) predictions become today's labeling queue. Data collection stops being a phase and
becomes a *subsystem*.

## 3. Class imbalance — the accuracy trap

Fraud, defects, urgent tickets: the interesting class is rare. With a 99:1 split,
**accuracy is a broken instrument**:

In [ ]:
def imbalanced(n, seed, pos_rate=0.01):
    r = np.random.default_rng(seed)
    y = (r.random(n) < pos_rate).astype(int)
    X = r.normal(0, 1.3, (n, 2)) + np.where(y[:, None] == 1, [1.8, 1.4], [0, 0])
    return X, y

X_tr, y_tr = imbalanced(30_000, seed=1)
X_te, y_te = imbalanced(20_000, seed=2)

dummy = DummyClassifier(strategy="most_frequent").fit(X_tr, y_tr)
naive = LogisticRegression().fit(X_tr, y_tr)
balanced = LogisticRegression(class_weight="balanced").fit(X_tr, y_tr)

# Undersampling: keep all positives, a matched slice of negatives.
pos, neg = np.where(y_tr == 1)[0], np.where(y_tr == 0)[0]
keep = np.concatenate([pos, rng.choice(neg, size=len(pos) * 3, replace=False)])
under = LogisticRegression().fit(X_tr[keep], y_tr[keep])

rows = []
for name, model in [("always-negative", dummy), ("naive LR", naive),
                    ("class-weighted", balanced), ("undersampled 3:1", under)]:
    pred = model.predict(X_te)
    rows.append((name, accuracy_score(y_te, pred),
                 precision_score(y_te, pred, zero_division=0),
                 recall_score(y_te, pred)))
report = pd.DataFrame(rows, columns=["model", "accuracy", "precision", "recall"])
print(report.round(3).to_string(index=False))

acc = report.set_index("model")
assert acc.loc["always-negative", "accuracy"] > 0.98, \
    "the useless model 'wins' on accuracy - the trap, quantified"
assert acc.loc["always-negative", "recall"] == 0.0
assert acc.loc["class-weighted", "recall"] > acc.loc["naive LR", "recall"] + 0.15, \
    "reweighting the loss finds the positives the naive fit ignored"
assert acc.loc["class-weighted", "accuracy"] < acc.loc["naive LR", "accuracy"], \
    "...and pays for them with false alarms: accuracy DOWN, usefulness UP"

In [ ]:
fig, ax = plt.subplots()
for name, model in [("naive LR", naive), ("class-weighted", balanced)]:
    p, r, _ = precision_recall_curve(y_te, model.predict_proba(X_te)[:, 1])
    ax.plot(r, p, label=name)
ax.set_xlabel("recall")
ax.set_ylabel("precision")
ax.set_title("Judge rare-class models in precision-recall space")
ax.legend()
plt.show()

The toolbox, in the order to try it: (1) **use the right metric** — precision/recall/PR
curves per class (ROC-AUC flatters imbalanced models: its false-positive-rate denominator
is the huge negative class); (2) **move the threshold** with the cost math of nb 13 —
often sufficient alone; (3) **reweight the loss** (`class_weight`) — like telling the
optimizer each fraud costs 99 non-frauds; (4) **resample** — undersample the majority
(cheap, discards data) or oversample/synthesize the minority (SMOTE-style); (5) **collect
more positives** — active learning aimed at the rare class. Note what stayed constant
above: the *data's* difficulty. These techniques trade error types; only better features
or more real positives reduce total error.

## 4. Augmentation — declaring invariances as data

Augmentation manufactures examples by applying transformations you *know* preserve the
label — flips and rotations for images, synonym swaps for text. When it works, it is
because the collection process **undersampled a true invariance** and you filled the gap
with compute instead of collection.

The miniature: the true label depends only on a point's *radius* (rotation-invariant), but
the sensor that collected training data only ever saw a narrow wedge of angles — a
collection bias, like a camera that only photographed dogs facing left. Deployment sees
the full circle:

In [ ]:
def ring_data(n, seed, angle_range):
    r = np.random.default_rng(seed)
    radius = r.uniform(0.2, 1.8, n)
    angle = r.uniform(*angle_range, n)
    X = np.c_[radius * np.cos(angle), radius * np.sin(angle)]
    return X, (radius > 1.0).astype(int)          # label = a rotation-invariant fact

X_tr2, y_tr2 = ring_data(400, 42, (0, np.pi / 4))       # collected in one wedge
X_te2, y_te2 = ring_data(4_000, 43, (0, 2 * np.pi))     # deployed everywhere

plain = DecisionTreeClassifier(random_state=0).fit(X_tr2, y_tr2)
acc_plain = accuracy_score(y_te2, plain.predict(X_te2))

# Augment: 8 randomly-ROTATED copies of every training point, same labels -
# the invariance we know to be true, stated as data.
copies = []
for _ in range(8):
    theta = rng.uniform(0, 2 * np.pi, len(X_tr2))
    c, s = np.cos(theta), np.sin(theta)
    copies.append(np.c_[X_tr2[:, 0] * c - X_tr2[:, 1] * s,
                        X_tr2[:, 0] * s + X_tr2[:, 1] * c])
augmented = DecisionTreeClassifier(random_state=0).fit(
    np.vstack([X_tr2] + copies), np.tile(y_tr2, 9))
acc_aug = accuracy_score(y_te2, augmented.predict(X_te2))

print(f"full-circle deployment accuracy - wedge-trained: {acc_plain:.3f}, "
      f"rotation-augmented: {acc_aug:.3f}")
assert acc_plain < 0.75, "the model learned the wedge, not the concept"
assert acc_aug > 0.95, "augmentation taught the invariance the data never showed"

Two rules, and one warning. Rules: the transform must **truly preserve the label** (know
your domain — flipping a chest X-ray left-right is fine; flipping a "9" makes a "6"), and
it should **cover deployment conditions** the training set misses (that was the wedge).
The warning, learned honestly while building this notebook: *generic noise injection is
not automatically augmentation* — perturbing points near a decision boundary manufactures
mislabeled examples, and measured accuracy can go **down**. Augment invariances you can
defend, and measure (a nb 17 evaluation) rather than assume.

## 5. Exercises

1. **Reservoir, proved.** Show by induction why every item ends in the reservoir with
   probability exactly k/n. Then: your stream is click events and you want a sample
   *weighted by dwell time* — name the algorithm family and its one-line trick.
2. **The 1% log.** To cut costs, you log 1% of requests as training data, but 100% of
   requests that error. Name two biases this bakes into a model trained on the logs, and a
   correction for each (hint: §1.1 and importance weights).
3. **Clicks as labels.** For search ranking, "clicked = relevant" is a free natural label.
   List three ways it lies (position bias, presentation bias, no-click ambiguity) and one
   mitigation each. Which nb 19 phenomenon does training on your own ranking's clicks
   create?
4. **Beyond majority vote.** Two of your LFs are near-duplicates (both keyword lists) and
   one is high-precision regex. Why does majority voting overweight the duplicated signal,
   and how can a label model use *agreement patterns on unlabeled data* to learn per-LF
   accuracies and correlations without any ground truth?
5. **Fraud at 1:5000.** Combine this notebook into a plan: sampling for the training set,
   labeling strategy for new fraud patterns, imbalance handling, and the two metrics on
   the exec dashboard (one statistical, one in dollars — nb 13).

### Solutions

**1.** Induction: item i enters with k/i; a resident survives round i with
1 − (k/i)(1/k) = (i−1)/i; telescoping the survival product from entry to n gives k/n for
every item. Weighted: **A-Res / A-ExpJ** (Efraimidis–Spirakis): key = u^(1/w) per item
(u ~ U(0,1)), keep the top-k keys — one pass, weights honored.

**2.** (a) Error-enriched data: the model learns "the world fails 100× more than it
does" — miscalibrated probabilities; fix with **importance weights** (weight normal
traffic ×100, or downweight errors) so the training distribution matches reality.
(b) Segment bias: 1% uniform of *requests* oversamples heavy users relative to *users*
(§1.1's power-user trap); fix by sampling per-user (stratify the sampling unit to match
the prediction unit).

**3.** Position bias: top results get clicked regardless of relevance → mitigate with
randomization/interleaving or inverse-propensity weighting by position. Presentation
bias: pretty snippets win clicks → include presentation features so the model can
attribute the effect. No-click ambiguity: unseen ≠ irrelevant → treat only *examined*
non-clicks as negatives (click models). Training on clicks your own ranker produced is a
**degenerate feedback loop** — nb 19 builds it and watches it spiral.

**4.** Duplicate LFs vote twice for one signal — majority arithmetic treats correlated
votes as independent evidence. A label model treats true label as latent: LF accuracies
and pairwise correlations leave fingerprints in the *observed agreement matrix* (two
90%-accurate independent LFs agree ~82% of the time; deviations reveal correlation), so
you can fit accuracies/correlations from unlabeled agreement statistics alone, then
weight votes accordingly (Snorkel's generative model).

**5.** Sampling: keep **all** fraud ever seen + stratified/undersampled legitimate
traffic (≈10:1), importance-weight to correct probabilities. Labeling: chargebacks as
delayed natural labels + analyst adjudication queue fed by **active learning** on
low-confidence/high-amount cases + LFs encoding analyst heuristics for instant coverage
of new patterns. Modeling: class weights + nb 13 threshold economics. Dashboard: PR-AUC
(statistical), and **$ prevented minus $ blocked-in-error** at the operating threshold
(nb 13's expected-profit curve, reported monthly).

## 6. Takeaways

- **Bias beats volume**: convenience samples stay wrong forever; stratify what must be
  represented; reservoir-sample what streams; sample the *unit you predict about*.
- Labels are designed and disagreement is signal: measure agreement, adjudicate
  systematically, and prefer *executable* labeling rules (LFs) — relabeling becomes a
  refold, not a project.
- Under a budget: weak supervision buys volume, active learning buys placement. Both beat
  spending the same money uniformly.
- Imbalance: accuracy lies; live in precision-recall space; reweight/resample/threshold
  by cost — and remember these reshape errors, not reduce them.
- Augmentation = declaring invariances as data; label-preserving and
  deployment-resembling, or it backfires.

**Further reading.** DMLS ch. 4; Ratner et al., *Snorkel: Rapid Training Data Creation
with Weak Supervision* (VLDB 2017); Settles, *Active Learning Literature Survey* (2009).

**Next:** [16 · Feature engineering](16-feature-engineering.ipynb) — the examples exist;
now the columns. Including the ways they secretly cheat.